# 16 — DocTR OCR Engine

Implements and benchmarks DocTR as a drop-in replacement for Tesseract in the
LayoutLMv3 field extraction pipeline.

| | |
|---|---|
| **Improvement 3** | DocTR OCR — deep-learning engine with cleaner word segmentation |
| **Failure mode addressed** | Tesseract merges tokens that should be separate: `to:Nicole`, `Date:2024-01-01` — causing the recipient fallback and date extraction to fail |
| **Drop-in replacement** | `ocr_image_doctr()` returns the same `(words, boxes)` format as `ocr_image_tesseract()` — no other pipeline changes needed |
| **Fallback** | If `python-doctr` is not installed, silently falls back to Tesseract — never crashes |
| **Module** | `src/extraction_improvements.py` |
| **No retraining** | LayoutLMv3 weights read-only |
| **No generative AI** | Both OCR engines and LayoutLMv3 are discriminative only |

**Workflow:**
1. Run Section 0 (setup + install doctr) after every kernel restart
2. Section 1 shows the DocTR function and its output format
3. Section 2 compares Tesseract vs DocTR token-by-token on 3 examples
4. Section 3 analyses merged-token frequency in Tesseract output
5. Section 4 benchmarks both engines end-to-end through the full pipeline

## 0. Setup — run after every kernel restart

In [1]:
import subprocess, sys

print('Installing python-doctr if not already present...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'python-doctr', '-q'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('python-doctr OK')
else:
    print('pip install failed — will fall back to Tesseract automatically')
    print(result.stderr[:400])

Installing python-doctr if not already present...
python-doctr OK


In [2]:
import os, time

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

t0 = time.time()
from transformers import LayoutLMv3ForTokenClassification, LayoutLMv3Processor
print('transformers imported in', round(time.time() - t0, 2), 'sec')
print('Python:', sys.executable)

/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/.venv311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers imported in 3.83 sec
Python: /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/.venv311/bin/python


In [3]:
import json
import torch
from pathlib import Path
from PIL import Image
from datasets import load_from_disk

PROJECT_ROOT = Path('..').resolve()
DATASET_DIR  = PROJECT_ROOT / 'data' / 'processed' / 'layoutlmv3_dataset'
CKPT_PATH    = PROJECT_ROOT / 'models' / 'experimental' / 'layoutlmv3_fatura' / 'best_checkpoint'
MAX_LENGTH   = 512

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else 'cpu'
)

with open(DATASET_DIR / 'label2id.json') as f:
    label2id = json.load(f)
with open(DATASET_DIR / 'id2label.json') as f:
    id2label = {int(k): v for k, v in json.load(f).items()}

FIELD_ORDER = [
    'INVOICE_NUMBER', 'INVOICE_DATE', 'DUE_DATE',
    'ISSUER_NAME', 'RECIPIENT_NAME', 'TOTAL_AMOUNT'
]
CLEAN_KEY = {
    'INVOICE_NUMBER': 'invoice_number', 'INVOICE_DATE': 'invoice_date',
    'DUE_DATE': 'due_date', 'ISSUER_NAME': 'issuer_name',
    'RECIPIENT_NAME': 'recipient_name', 'TOTAL_AMOUNT': 'total_amount',
}

raw_dataset = load_from_disk(str(DATASET_DIR))

print(f'Device     : {DEVICE}')
print(f'Checkpoint : {CKPT_PATH}')
print(f'Dataset    : {raw_dataset}')

Device     : mps
Checkpoint : /Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/document-classification/models/experimental/layoutlmv3_fatura/best_checkpoint
Dataset    : DatasetDict({
    train: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 1734
    })
    val: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 371
    })
    test: Dataset({
        features: ['image_path', 'words', 'bboxes', 'ner_tags'],
        num_rows: 372
    })
})


In [4]:
processor = LayoutLMv3Processor.from_pretrained(
    str(CKPT_PATH),
    apply_ocr=False,
    use_fast=True,
    local_files_only=True,
)

model = LayoutLMv3ForTokenClassification.from_pretrained(
    str(CKPT_PATH),
    id2label=id2label,
    label2id=label2id,
    local_files_only=True,
)
model.to(DEVICE)
model.eval()

print('Model loaded OK  |  device:', DEVICE)

The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.
Loading weights: 100%|██████████| 216/216 [00:00<00:00, 8109.00it/s]


Model loaded OK  |  device: mps


In [5]:
SRC_DIR = str(PROJECT_ROOT / 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from invoice_cleaner import InvoiceCleaner
from extraction_improvements import (
    ocr_image_tesseract,
    ocr_image_doctr,
    ocr_image,
    sort_reading_order,
    extract_with_confidence_gating,
)

cleaner = InvoiceCleaner()

# Check if doctr loaded successfully
try:
    from doctr.models import ocr_predictor
    _DOCTR_AVAILABLE = True
    print('DocTR available    : YES')
except ImportError:
    _DOCTR_AVAILABLE = False
    print('DocTR available    : NO — will use Tesseract fallback throughout')
print('InvoiceCleaner     : OK')
print('extraction_improvements : OK')

DocTR available    : YES
InvoiceCleaner     : OK
extraction_improvements : OK


## 1. DocTR OCR Function

`ocr_image_doctr()` is a drop-in replacement for `ocr_image_tesseract()`.  Both
return `(words, boxes)` with boxes normalised to [0, 1000].  The rest of the
pipeline — sort_reading_order, get_raw_predictions_with_confidence, InvoiceCleaner
— requires no changes.

If `python-doctr` is not installed, `ocr_image_doctr()` automatically falls back
to Tesseract with a `RuntimeWarning`.

In [6]:
import warnings

# Demo: run both engines on test example 0
SPLIT = 'test'
example = raw_dataset[SPLIT][0]
image   = Image.open(example['image_path']).convert('RGB')
print(f'Image: {Path(example["image_path"]).stem}')
print(f'Size : {image.size}')
print()

# Tesseract
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    tess_words, tess_boxes = ocr_image_tesseract(image)
print(f'Tesseract tokens : {len(tess_words)}')
print(f'First 8: {tess_words[:8]}')
print()

# DocTR (falls back automatically if not installed)
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    doctr_words, doctr_boxes = ocr_image_doctr(image)
    if w:
        for warning in w:
            print(f'WARNING: {warning.message}')

print(f'DocTR tokens     : {len(doctr_words)}')
print(f'First 8: {doctr_words[:8]}')
print()
print(f'Output formats match: '
      f'{isinstance(doctr_words, list) and isinstance(doctr_boxes, list)}')
print(f'All boxes in [0,1000]: '
      f'{all(0 <= v <= 1000 for b in doctr_boxes for v in b)}')

Image: Template1_Instance189
Size : (595, 841)

Tesseract tokens : 109
First 8: ['Address:16424', 'Timothy', 'Mission', 'Markville,', 'AK', '58294', 'US', '‘www.ThompsonandSons.org']



65815552it [00:01, 52351773.98it/s]                               


63303680it [00:00, 72099548.35it/s]                              


DocTR tokens     : 113
First 8: ['TAX', 'INVOICE', 'D0', 'Date:', '29-Apr-2012', 'Due', 'Date', ':']

Output formats match: True
All boxes in [0,1000]: True


## 2. Token Comparison — Tesseract vs DocTR Side by Side

For each test example, prints the full token lists from both engines side by
side, highlighting positions where the token sets differ.

In [7]:
import re
SPLIT    = 'test'
N_EXAMPLES = 3

def _is_merged(token: str) -> bool:
    """Detect tokens that merge a label and a value without a space.
    E.g. 'to:Nicole', 'Date:2024', 'Invoice#123', 'Bill_To_Name'.
    """
    # Contains colon/hash/underscore followed immediately by non-whitespace
    # AND has at least one letter before and after the separator
    if re.search(r'[A-Za-z][:#_][A-Za-z0-9]', token):
        return True
    # Camel-case boundary between a label word and a value
    # e.g. 'BillTo', 'InvoiceNumber'
    if re.search(r'[a-z][A-Z]', token) and len(token) > 8:
        return True
    return False

for i in range(N_EXAMPLES):
    example = raw_dataset[SPLIT][i]
    image   = Image.open(example['image_path']).convert('RGB')

    tess_words, _ = ocr_image_tesseract(image)
    doctr_words, _ = ocr_image_doctr(image)

    print(f"\n{'='*70}")
    print(f"  TEST {i}: {Path(example['image_path']).stem}")
    print(f"  Tesseract tokens: {len(tess_words)}  |  DocTR tokens: {len(doctr_words)}")
    print('='*70)
    print(f"  {'#':>3}  {'TESSERACT':^30}  {'DOCTR':^30}")
    print(f"  {'---':>3}  {'-'*30}  {'-'*30}")

    n_show = min(30, max(len(tess_words), len(doctr_words)))
    for j in range(n_show):
        t = tess_words[j] if j < len(tess_words) else ''
        d = doctr_words[j] if j < len(doctr_words) else ''
        t_flag = ' [M]' if _is_merged(t) else ''
        d_flag = ' [M]' if _is_merged(d) else ''
        t_d = (repr(t)[:26] + '..') if len(repr(t)) > 28 else repr(t)
        d_d = (repr(d)[:26] + '..') if len(repr(d)) > 28 else repr(d)
        marker = '<<' if t != d else '  '
        print(f"  {j:>3}  {t_d+t_flag:<30}  {d_d+d_flag:<30} {marker}")
    if n_show < max(len(tess_words), len(doctr_words)):
        print(f'  ... (showing first {n_show} of {max(len(tess_words),len(doctr_words))} tokens)')

print('\n[M] = merged token (label+value in one token with no space)')


  TEST 0: Template1_Instance189
  Tesseract tokens: 109  |  DocTR tokens: 113
    #            TESSERACT                         DOCTR             
  ---  ------------------------------  ------------------------------
    0  'Address:16424' [M]             'TAX'                          <<
    1  'Timothy'                       'INVOICE'                      <<
    2  'Mission'                       'D0'                           <<
    3  'Markville,'                    'Date:'                        <<
    4  'AK'                            '29-Apr-2012'                  <<
    5  '58294'                         'Due'                          <<
    6  'US'                            'Date'                         <<
    7  '‘www.ThompsonandSons.org' [M]  ':'                            <<
    8  '(GSTIN:'                       '07-Aug-2010'                  <<
    9  '12345670'                      'PO'                           <<
   10  '00070007'                      'Number'    

## 3. Merged Token Analysis

Counts how many tokens Tesseract merges (label + value with no space separator)
across a set of examples.  These are the exact tokens that cause the
recipient-name fallback and date-extraction to fail.

In [8]:
N_ANALYSE = 10
SPLIT     = 'test'

tess_merged_counts  = []
doctr_merged_counts = []
tess_merged_examples = []   # (example_idx, token)

for i in range(N_ANALYSE):
    example = raw_dataset[SPLIT][i]
    image   = Image.open(example['image_path']).convert('RGB')
    tess_words, _ = ocr_image_tesseract(image)
    doctr_words, _ = ocr_image_doctr(image)
    t_merged = [w for w in tess_words if _is_merged(w)]
    d_merged = [w for w in doctr_words if _is_merged(w)]
    tess_merged_counts.append(len(t_merged))
    doctr_merged_counts.append(len(d_merged))
    for w in t_merged:
        tess_merged_examples.append((i, w))

print(f'Merged-token analysis over {N_ANALYSE} {SPLIT} examples')
print(f"{'='*50}")
print(f"  {'METRIC':<35}  {'TESSERACT':>10}  {'DOCTR':>8}")
print(f"  {'-'*35}  {'-'*10}  {'-'*8}")
print(f"  {'Total merged tokens':<35}  {sum(tess_merged_counts):>10}  {sum(doctr_merged_counts):>8}")
print(f"  {'Mean per example':<35}  {sum(tess_merged_counts)/N_ANALYSE:>10.1f}  {sum(doctr_merged_counts)/N_ANALYSE:>8.1f}")
print(f"  {'Examples with >= 1 merged':<35}  {sum(1 for c in tess_merged_counts if c > 0):>10}  {sum(1 for c in doctr_merged_counts if c > 0):>8}")

if tess_merged_examples:
    print(f'\nTesseract merged tokens (all {len(tess_merged_examples)} occurrences):')
    for ex_idx, tok in tess_merged_examples:
        print(f'  example {ex_idx}: {repr(tok)}')
else:
    print('\nNo merged tokens detected in Tesseract output for these examples.')

Merged-token analysis over 10 test examples
  METRIC                                TESSERACT     DOCTR
  -----------------------------------  ----------  --------
  Total merged tokens                          46        48
  Mean per example                            4.6       4.8
  Examples with >= 1 merged                    10        10

Tesseract merged tokens (all 46 occurrences):
  example 0: 'Address:16424'
  example 0: '‘www.ThompsonandSons.org'
  example 0: 'to:Shelly'
  example 0: 'Email:christina14@example.org'
  example 0: 'Site:http://www.gomez.com!'
  example 0: 'Note:Total'
  example 0: '‘SUB_TOTAL'
  example 0: 'TAX:VAT'
  example 1: '‘Address:02850'
  example 1: 'SUB_TOTAL'
  example 2: 'to:Danie!'
  example 2: 'Email:frichardson@example.org'
  example 2: 'Site:httpsilIberger-bailey.com/'
  example 2: 'SUB_TOTAL'
  example 2: 'Address:86143'
  example 3: 'BALANCE_DUE'
  example 3: '‘Address:840'
  example 3: 'Email:cindyglass@example.net'
  example 3: 'Site:https:/ip

## 4. End-to-End Benchmark

Runs both OCR engines through the full improved pipeline (reading order sort +
confidence-gated extraction) on 5 test examples.  Compares:
- Number of word tokens
- Merged token count
- Final extraction quality (which fields are populated)

In [9]:
N_BENCH   = 5
SPLIT     = 'test'
THRESHOLD = 0.70

for i in range(N_BENCH):
    example = raw_dataset[SPLIT][i]
    image   = Image.open(example['image_path']).convert('RGB')

    results = {}
    token_counts = {}
    merged_counts = {}

    for engine in ('tesseract', 'doctr'):
        w, b = ocr_image(image, engine=engine)
        token_counts[engine]  = len(w)
        merged_counts[engine] = sum(1 for tok in w if _is_merged(tok))
        w, b = sort_reading_order(w, b)
        out = extract_with_confidence_gating(
            image, w, b, model, processor, DEVICE, id2label, cleaner,
            model_threshold=THRESHOLD, ocr_text=' '.join(w), max_length=MAX_LENGTH
        )
        results[engine] = out

    print(f"\n{'='*80}")
    print(f"  TEST {i}: {Path(example['image_path']).stem}")
    print('='*80)
    print(f"  {'METRIC':<25}  {'TESSERACT':>12}  {'DOCTR':>10}")
    print(f"  {'-'*25}  {'-'*12}  {'-'*10}")
    print(f"  {'Word tokens':<25}  {token_counts['tesseract']:>12}  {token_counts['doctr']:>10}")
    print(f"  {'Merged tokens':<25}  {merged_counts['tesseract']:>12}  {merged_counts['doctr']:>10}")
    print()
    print(f"  {'FIELD':<20}  {'TESSERACT RESULT':^28}  DOCTR RESULT")
    print(f"  {'-'*20}  {'-'*28}  {'-'*28}")
    for field in FIELD_ORDER:
        ck    = CLEAN_KEY[field]
        t_val = results['tesseract'].get(ck, '') or '—'
        d_val = results['doctr'].get(ck, '')     or '—'
        marker = ' <<' if t_val != d_val else ''
        t_d = (t_val[:26] + '..') if len(t_val) > 28 else t_val
        d_d = (d_val[:26] + '..') if len(d_val) > 28 else d_val
        print(f"  {field:<20}  {t_d:<28}  {d_d}{marker}")

print('\n<< = DocTR produced a different result than Tesseract')


  TEST 0: Template1_Instance189
  METRIC                        TESSERACT       DOCTR
  -------------------------  ------------  ----------
  Word tokens                         109         113
  Merged tokens                         8           9

  FIELD                       TESSERACT RESULT        DOCTR RESULT
  --------------------  ----------------------------  ----------------------------
  INVOICE_NUMBER        —                             —
  INVOICE_DATE          29-Apr-2012                   29-Apr-2012
  DUE_DATE              07-Aug-2010                   07-Aug-2010
  ISSUER_NAME           —                             —
  RECIPIENT_NAME        Shelly Rodriguez Markville..  Shelly Rodriguez Markville..
  TOTAL_AMOUNT          134.41 USD                    134.41 USD

  TEST 1: Template38_Instance29
  METRIC                        TESSERACT       DOCTR
  -------------------------  ------------  ----------
  Word tokens                          97          99
  Merged toke